# Detecting AI-generated choreography in Dance Dance Revolution step charts

## **Identify the predictive task**

In [ ]:
'''
Identify the predictive task you will study.
○ Describe how you will evaluate your model at this predictive task
○ What relevant baselines can be used for comparison
○ How you will assess the validity of your model’s predictions?
Make sure to select a task and models that are relevant to the course content; if you
want to try out models you’ve seen in other classes that’s fine, but you should still
implement models from this class as baselines / comparison points.
'''

In this notebook, we will be analyzing step charts from dance dance revolution. Specifically, for each of the songs in our dataset, we have a human generated and AI generated step chart. Our predictive task is to create a binary classifier that identifies whether the step chart is human or AI generated.

To evaluate our model's performance, we will use accuracy. That is, what fraction of the step charts are classified correctly by the model. We will have 2 baselines:

1. Random Guessing: This will achieve 50% accuracy as it is equally likely to choose between 2 choices
2. Logistic Regression: A model that we covered in class.

To assess the validiity of our model's predictions, we have used a couple of strategies:

1. Split strategy: We will use a 80/20 hold out split by song to prevent leakage
2. Primary Metric: Accuracy will be used as the primary metric.
3. Secondary Metrics: Precision, recall, F1, ROC-AUC will be used as additional metrics.
4. Cross Validation: We will be using a 5 fold stratified Cross Validation on the training set for ROC-AUC stability. 
5. Error Inspection: Confusion Matrices will also help assess validity.

## **EDA**

In [ ]:
'''
Context: Where does your dataset come from? What is it for, how was it
collected, etc.?
○ Discussion: Report how you processed the data (or how it was already
processed);
○ Code: Support your analysis with tables, plots, statistics, etc.
'''

## Context

Our dataset consists of paired audio files and step-chart annotations used for Dance Dance Revolution–style rhythm gameplay. These data originate from the **Dance Dance Convolution (DDC)** project by Donahue et al., available at:
**[https://github.com/chrisdonahue/ddc](https://github.com/chrisdonahue/ddc)**

The original DDC dataset was assembled from publicly available StepMania song packs containing human-authored `.sm` files (step charts, timing metadata, difficulty information) paired with their corresponding `.ogg` audio tracks. These data were collected to support research on sequence-to-sequence generative modeling—specifically, the task of producing human-like dance step charts conditioned on music. The DDC dataset has since become a standard benchmark for rhythm-game procedural content generation.

For our work, we rely on the cleaned and structured version of these human-created charts provided in the DDC repository.

## Discussion: Data Processing

To construct the dataset required for our classifier, we generated a new corpus of **AI-produced step charts** using the official DDC inference server and paired them with the **human-authored charts** included in the repository.

Although DDC provides inference scripts, we wrote our own batch-processing Python client to systematically run the DDC model on all tracks. This approach allowed us to control output formatting and ensure consistency between AI and human `.sm` files. Our script iterates through all `.ogg` files in `ddc/data/raw`, queries the Dockerized DDC server for multiple difficulty levels, and reconstructs unified `.sm` files that mimic the structure of human-authored charts.

### Processing steps:

1. **Header extraction / normalization**
   For each song, we extract the header from the human-authored `.sm` file when available. If a human chart does not exist, we generate a fallback StepMania-style header. This ensures that AI-generated charts use the same metadata format as human ones.

2. **AI chart generation**
   Using the DDC server, we request charts across several difficulty levels (Beginner, Easy, Medium, Hard, Challenge). The server returns a ZIP archive containing an `.sm` file, from which we extract the `#NOTES` blocks.

3. **Reconstruction into a unified `.sm` file**
   For each song, all generated difficulties are combined into a single `.sm` file using the normalized header and properly rewritten difficulty labels. This results in AI charts that are structurally parallel to human charts.

4. **Dataset organization**
   We store AI-generated `.sm` files in a dedicated folder (`ai_generated_sm/`), while the human `.sm` files are stored in (`human_generated_sm/`). This clean separation enables us to build a binary classification dataset distinguishing **AI-generated** vs. **human-authored** charts.

## **Modeling**

In [ ]:
'''
Context: How do you formulate your task as an ML problem, e.g. what are the
inputs, outputs, and what is being optimized? What models are appropriate for
the task?
○ Discussion: Discuss the advantages and disadvantages of different modeling
approaches (complexity, efficiency, challenges in implementation, etc.)
○ Code: Walk through your code, explaining architectural choices and any
implementation details.
'''

## **Evaluation**

In [ ]:
'''
Context: How should your task be evaluated? Can you justify why your particular
metrics are more appropriate than others?
○ Discussion: What are some baselines (trivial or otherwise) for your task? How
do you demonstrate that your method is better than these methods?
○ Code: Walk through the implementation of your evaluation protocol, and support
your evaluation with tables, plots, statistics, etc.
'''

## **Related Work**

In [ ]:
'''
How has this dataset (or similar datasets) been used before?
○ How has prior work approached the same (or similar) tasks?
○ How do your results match or differ from what has been reported in related work?
'''

### Use of This Dataset in Prior Work

The dataset used in our project, paired music/audio features and corresponding dance
motion-capture sequences, is closely aligned with datasets used in prior computational
dance-generation research. In particular, the *Dance Dance Convolution (DDC)* dataset
(Donahue et al., 2017) is one of the most influential resources in this domain.  

DDC contains aligned pairs of audio inputs and choreographed step or motion sequences and
has been widely used to study multimodal sequence modeling, music-to-movement prediction, 
and automatic choreography generation.

### How Prior Work Has Approached Similar Tasks

Prior research typically approaches the music-to-dance task using:

1. **Sequence-to-Sequence Neural Models**  
   DDC introduced convolutional and recurrent neural architectures that map audio features
   to symbolic movement sequences. These models emphasize temporal alignment and beat tracking.

2. **Recurrent / LSTM-Based Models**  
   Early work (including DDC) used RNNs to capture rhythmic dependencies and model long-term 
   temporal structure.

3. **Conditional Generative Models**  
   Later work introduced Transformers, autoregressive models, or diffusion-based approaches
   to generate smoother, more realistic motion trajectories.

4. **Beat-Synchronous or Tempo-Aligned Pipelines**  
   Many pipelines normalize or quantize the data so the model primarily learns rhythm↔movement
   relationships.

Our assignment follows a simplified version of these paradigms: extracting audio features,
learning a mapping into movement output space, and evaluating via accuracy metrics.

### Comparison Between Our Results and Related Work
